# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library and Python tools for data analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We will display each record set's `@id`, field `@id`s, and columns for orientation.

In [ ]:
# List all record sets and their fields
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rset in record_sets:
    print(f"- Record set @id: {rset['@id']}")
    print(f"  Name: {rset.get('name', '(no name)')}")
    fields = rset.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id', str(f))}")
        else:
            print(f"    - {f}")
    # Also list columns for each field if available
    for f in fields:
        field = f if isinstance(f, dict) else dataset.get_by_id(f)
        if field:
            columns = field.get('column', [])
            if not isinstance(columns, list):
                columns = [columns]
            if columns:
                print("      Columns:")
                for c in columns:
                    if isinstance(c, dict):
                        print(f"        * {c.get('@id', str(c))}")
                    else:
                        print(f"        * {c}")
    print('')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All record set and field `@id`s are used explicitly for referencing.

In [ ]:
# Extract data from each record set
record_set_ids = [r['@id'] for r in dataset.record_sets]
print("Record sets available:", record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from '{rs_id}'. Columns: {df.columns.tolist()}")

# As an example, show the columns and first records from the first record set (if present)
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in the first record set '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We will select a numeric field (referenced by its `@id`), filter rows based on a threshold, normalize the values, and group by another categorical field if available.

*All field and column references use their explicit `@id`.*

In [ ]:
# For demonstration, let's auto-detect a numeric field and a group (categorical) field.
from pandas.api.types import is_numeric_dtype

current_rs_id = record_set_ids[0]  # Use the first record set as example
df = dataframes[current_rs_id]

# Find a numeric field by @id
numeric_field_id = None
for col in df.columns:
    if is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    print("No numeric field found in the first record set.")
else:
    print(f"Numeric field selected (by @id): {numeric_field_id}")
    threshold = np.percentile(df[numeric_field_id].dropna(), 75) if not df[numeric_field_id].empty else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by a categorical field if available
    group_field_id = None
    for col in df.columns:
        if col == numeric_field_id:
            continue
        if df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col]):
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with the group field if possible.

In [ ]:
# Plot histogram and boxplot for the numeric field
if numeric_field_id and not df[numeric_field_id].empty:
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    df[numeric_field_id].hist(bins=20)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")

    plt.subplot(1,2,2)
    df.boxplot(column=numeric_field_id)
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()
    
    # If we grouped by a field, make a bar plot
    if 'grouped_df' in locals() and group_field_id:
        grouped_df.plot(kind='bar', x=group_field_id, y=numeric_field_id, legend=False)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to:
* Load and inspect a Croissant-structured dataset using `mlcroissant`, referencing all entities by their `@id`s
* Extract tabular data from record sets and examine the fields and columns structure
* Perform basic EDA, filtering, normalization, and grouping with explicit use of field `@id`s
* Visualize numeric field distributions and group differences

This approach ensures reproducibility, clarity, and full traceability to the dataset schema for downstream analysis and reporting.